# Fast L² bandwidth distributions: sample size and lambda

Start with the Figure 3.6 baseline: **ten separate bandwidth distributions** for $n=100,200,\ldots,1000$, at **$\lambda=1$**. Each panel shows **only spectrally chosen bandwidths**. There are no ISE or Q-hat distributions and no reference lines.

Use the existing FFT-based Gaussian KDE implementation with

$$D_{ij}=\frac12\|\widehat f_{h_i}-\widehat f_{h_j}\|_2^2,\qquad S_{ij}=\exp(-\lambda D_{ij}).$$

Select the bandwidth at the largest Perron-vector component. Keep the original L² experiment's geometric grid, sampling seed, and nested samples. The default thesis profile uses $M=1000$ repetitions and $N=201$ candidates on $[0.01,2]$, with the periodic FFT interval $[-12,12)$ and 8,192 spatial points.

**Run the cells in order.** `LAMBDA_VALUES = [1.0]` gives the requested baseline first. To compare larger lambda values later, extend this list, for example to `[1.0, 10.0, 100.0, 1000.0]`, and rerun from the settings cell. Each lambda receives its own ten-panel figure. All lambdas share the same samples, and the FFT divergence matrix is calculated once per sample and reused across lambdas. Checkpoints and exports are kept separately for each complete configuration.

Sources: [Figure 3.6 / increasing-sample-size experiment, `fig:distribution_of_chosen_bandwidths_for_increasing_sample_size`](../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [original fast L² experiment, section 3 / `beta2_n_sweep_*`](spectral_scale_selection_kde.ipynb); [shared FFT divergence helper](kde_tools/fourier/similarities.py); [shared periodic KDE reconstruction](kde_tools/fourier/reconstruction.py).


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import sys
import time
from pathlib import Path

working_directory = Path.cwd().resolve()
code_directory = next(
    (candidate.resolve()
     for root in (working_directory, *working_directory.parents)
     for candidate in (root, root / "Code" / "Code_kde")
     if (candidate / "kde_tools" / "pdf_export.py").is_file()),
    None,
)
if code_directory is None:
    raise FileNotFoundError("Run this notebook from the project or Code/Code_kde folder.")
if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))
os.environ.setdefault("MPLCONFIGDIR", str(code_directory / ".matplotlib-cache"))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import PercentFormatter
from scipy.linalg import eigh as symmetric_eigh

from kde_tools.bandwidths import bandwidth_grid_linear_in_psi_beta
from kde_tools.fourier import beta2_divergence_matrix_fft
from kde_tools.pdf_export import enable_automatic_pdf_saving
from kde_tools.similarities import beta2_divergence_matrix

notebook_name = "fast_l2_bandwidth_distributions_vs_n_lambda"
pdf_output_directory = enable_automatic_pdf_saving(
    notebook_name, output_root=code_directory / "outputs" / "pdfs"
)
data_output_directory = code_directory / "outputs" / "data" / notebook_name
image_output_directory = code_directory / "outputs" / "images" / notebook_name
print(f"Automatic PDF exports: {pdf_output_directory}")


## Settings: lambda = 1 baseline

The default values match the fast L² sample-size experiment: $n=100,200,\ldots,1000$, $M=1000$, $N=201$, $h_{\min}=0.01$, $h_{\max}=2$, and seed `20260830` (the original notebook's `seed + 10 + 2`). One maximum-size Gaussian sample is drawn per repetition and its prefixes are used at every $n$.

The preview profile reduces the repetition count to 50 while keeping the same bandwidth and FFT grids. `LAMBDA_VALUES` controls subsequent comparisons; leave `[1.0]` for the first experiment.

Source: [the original fast L² notebook's setup, beta-2 settings, and increasing-sample-size cell](spectral_scale_selection_kde.ipynb).


In [ ]:
RUN_PROFILE = "thesis"  # "thesis" or "preview"
PROFILES = {
    "thesis": {"repetitions": 1000, "grid_size": 201},
    "preview": {"repetitions": 50, "grid_size": 201},
}
SAMPLE_SIZES = list(range(100, 1001, 100))
LAMBDA_VALUES = [1.0]  # Start here; later extend to [1.0, 10.0, 100.0, 1000.0].
REUSE_CHECKPOINTS = True
VALIDATE_FFT = True

settings = {
    **PROFILES[RUN_PROFILE],
    "sigma2": 1.0,
    "h_min": 0.01,
    "h_max": 2.0,
    "fft_half_width": 12.0,
    "fft_spatial_points": 8192,
    "seed": 20260830,
    "maximum_sample_size": max(SAMPLE_SIZES),
}
LAMBDA_VALUES = [float(value) for value in LAMBDA_VALUES]
if (not LAMBDA_VALUES or LAMBDA_VALUES[0] != 1.0
        or not np.all(np.isfinite(LAMBDA_VALUES))
        or np.any(np.diff(LAMBDA_VALUES) <= 0)):
    raise ValueError("Start LAMBDA_VALUES with 1.0 and use finite, strictly increasing values.")
if settings["repetitions"] < 1 or settings["grid_size"] < 2:
    raise ValueError("Repetitions must be positive and the bandwidth grid needs two points.")
if settings["sigma2"] <= 0 or settings["fft_half_width"] <= 0:
    raise ValueError("Variance and FFT half-width must be positive.")
if settings["fft_spatial_points"] < 2 or settings["fft_spatial_points"] % 2:
    raise ValueError("The FFT grid needs an even number of at least two points.")
h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=settings["h_min"], h_max=settings["h_max"], beta=2.0,
    grid_size=settings["grid_size"],
)
configuration = {
    "settings": settings, "sample_sizes": SAMPLE_SIZES,
    "lambda_values": LAMBDA_VALUES, "method": "fft_l2_spectral",
    "implementation_version": 1,
}
configuration_id = hashlib.sha256(json.dumps(configuration, sort_keys=True).encode()).hexdigest()[:12]
run_directory = data_output_directory / RUN_PROFILE / configuration_id
image_directory = image_output_directory / RUN_PROFILE / configuration_id
for directory in (run_directory, image_directory):
    directory.mkdir(parents=True, exist_ok=True)
(run_directory / "settings.json").write_text(
    json.dumps({"profile": RUN_PROFILE, **configuration}, indent=2) + "\n"
)
full_samples = np.random.default_rng(settings["seed"]).normal(
    0.0, np.sqrt(settings["sigma2"]),
    size=(settings["repetitions"], settings["maximum_sample_size"]),
)
print(f"Sample sizes: {SAMPLE_SIZES}; lambdas: {LAMBDA_VALUES}")
print(f"Profile: {RUN_PROFILE}; M={settings['repetitions']}; N={len(h_grid)}; "
      f"FFT spatial points={settings['fft_spatial_points']}")
print(f"Checkpoints and CSV data: {run_directory}")


## Fast L² calculation and spectral selection

Use the shared FFT divergence function directly. For each sample, reuse the resulting $D$ matrix for every configured lambda.

The eigenvector calculation follows the stable construction already used in `experiment_1_joint_N_lambda_limit.ipynb`: subtract the identity and rescale the off-diagonal similarity matrix before computing its largest eigenvector. This preserves the selected Perron peak and supports later runs with larger lambda values. Only the largest eigenvector is computed.

The original fast L² sample-size experiment selects the peak of the Doob stationary mass. For a symmetric similarity matrix, the shared implementation uses equal left and right Perron vectors, so that mass is proportional to the square of the Perron vector. **Consequently, both choices have the same maximizer** (this is the direct algebraic consequence of the shared implementation).

Sources: [stable eigenvector helpers in the joint-limit notebook](experiment_1_joint_N_lambda_limit.ipynb); [thesis subsection “Convergence to closest adjacent pair,” identity subtraction and positive rescaling](../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [`doob_stationary_mass` and Perron-vector implementations](kde_tools/similarities.py); [FFT divergence implementation](kde_tools/fourier/similarities.py).


In [ ]:
def fast_spectral_index(divergence, lambda_value):
    """Select the Perron peak using the existing stable off-diagonal construction."""
    divergence = np.asarray(divergence, dtype=float)
    if divergence.ndim != 2 or divergence.shape[0] != divergence.shape[1] or len(divergence) < 2:
        raise ValueError("Divergence must be a square matrix with at least two scales.")
    if (not np.all(np.isfinite(divergence)) or np.any(divergence < 0)
            or not np.allclose(divergence, divergence.T, atol=1e-12)):
        raise ValueError("Divergence must be finite, symmetric, and nonnegative.")
    lambda_value = float(lambda_value)
    if not np.isfinite(lambda_value) or lambda_value <= 0:
        raise ValueError("Lambda must be positive and finite.")
    # Same construction as stable_perron_vector_from_divergence in the joint-limit notebook.
    matrix = lambda_value * divergence
    np.fill_diagonal(matrix, np.inf)
    minimum_exponent = float(np.min(matrix))
    matrix -= minimum_exponent
    np.negative(matrix, out=matrix)
    np.exp(matrix, out=matrix)
    np.fill_diagonal(matrix, 0.0)
    number_of_scales = len(matrix)
    _, eigenvectors = symmetric_eigh(
        matrix, subset_by_index=[number_of_scales - 1, number_of_scales - 1],
        check_finite=False, overwrite_a=True, driver="evr",
    )
    vector = eigenvectors[:, 0]
    if vector.sum() < 0:
        vector = -vector
    return int(np.argmax(vector))


def validate_l2_checkpoint(indices, completed, repetitions, number_of_lambdas, grid_size):
    if indices.shape != (repetitions, number_of_lambdas):
        raise ValueError("Checkpoint shape differs from the current experiment.")
    if not np.issubdtype(indices.dtype, np.integer):
        raise ValueError("Checkpoint selections must be integer indices.")
    if not 0 <= completed <= repetitions:
        raise ValueError("Invalid checkpoint completion count.")
    if np.any(indices[:completed] < 0) or np.any(indices[:completed] >= grid_size):
        raise ValueError("Checkpoint contains invalid selected indices.")


def run_l2_point(sample_size, full_samples, h_grid, settings, lambda_values, output_directory, *, reuse=True):
    """One FFT divergence calculation per repetition, reused for all lambdas."""
    repetitions = int(settings["repetitions"])
    if full_samples.shape != (repetitions, settings["maximum_sample_size"]):
        raise ValueError("Sample array shape differs from the settings.")
    if not 1 <= sample_size <= full_samples.shape[1]:
        raise ValueError("Sample size is outside the generated sample range.")
    point_json = json.dumps({
        "settings": settings, "sample_size": int(sample_size),
        "lambda_values": list(lambda_values), "implementation_version": 1,
    }, sort_keys=True)
    point_directory = Path(output_directory) / f"n{int(sample_size)}"
    point_directory.mkdir(parents=True, exist_ok=True)
    checkpoint_path = point_directory / "checkpoint.npz"
    indices = np.full((repetitions, len(lambda_values)), -1, dtype=int)
    completed = 0

    def save_checkpoint():
        temporary = checkpoint_path.with_suffix(".npz.tmp")
        with temporary.open("wb") as stream:
            np.savez_compressed(
                stream, configuration_json=point_json, h_grid=h_grid,
                selected_indices=indices, completed=completed,
            )
        temporary.replace(checkpoint_path)

    if reuse and checkpoint_path.exists():
        with np.load(checkpoint_path, allow_pickle=False) as checkpoint:
            if str(checkpoint["configuration_json"].item()) != point_json:
                raise ValueError("L² checkpoint settings differ from the current configuration.")
            if not np.array_equal(checkpoint["h_grid"], h_grid):
                raise ValueError("L² checkpoint bandwidth grid differs from the current grid.")
            indices = checkpoint["selected_indices"].copy()
            completed = int(checkpoint["completed"].item())
    validate_l2_checkpoint(indices, completed, repetitions, len(lambda_values), len(h_grid))
    started = time.perf_counter()
    print(f"n={sample_size}: starting at {completed}/{repetitions}", flush=True)
    for repetition in range(completed, repetitions):
        divergence = beta2_divergence_matrix_fft(
            samples=full_samples[repetition, :int(sample_size)], h_grid=h_grid,
            half_width=settings["fft_half_width"], spatial_points=settings["fft_spatial_points"],
        )
        for column, lambda_value in enumerate(lambda_values):
            indices[repetition, column] = fast_spectral_index(divergence, lambda_value)
        completed = repetition + 1
        save_checkpoint()
        if completed == 1 or completed % 100 == 0 or completed == repetitions:
            print(f"n={sample_size}: {completed}/{repetitions}; elapsed this run: "
                  f"{time.perf_counter() - started:.1f}s", flush=True)
    validate_l2_checkpoint(indices, completed, repetitions, len(lambda_values), len(h_grid))
    counts = np.stack([
        np.bincount(indices[:, column], minlength=len(h_grid))
        for column in range(len(lambda_values))
    ])
    frequencies = counts / repetitions
    assert np.all(counts.sum(axis=1) == repetitions)
    assert np.allclose(frequencies.sum(axis=1), 1.0)
    return {"sample_size": int(sample_size), "selected_indices": indices,
            "counts": counts, "frequencies": frequencies}


## Small FFT validation

Compare the FFT divergence with the shared exact Gaussian formula on a small subset of bandwidths from the first sample. This check precedes the repeated simulation and does not compute any additional bandwidth-selection distribution.

Sources: [FFT validation in the original L² notebook](spectral_scale_selection_kde.ipynb); [exact Gaussian L² helper](kde_tools/similarities.py).


In [ ]:
if VALIDATE_FFT:
    validation_indices = np.unique(np.linspace(0, len(h_grid) - 1, min(len(h_grid), 20), dtype=int))
    validation_h_grid = h_grid[validation_indices]
    validation_samples = full_samples[0, :min(100, settings["maximum_sample_size"])]
    validation_exact = beta2_divergence_matrix(validation_samples, validation_h_grid)
    validation_fft = beta2_divergence_matrix_fft(
        validation_samples, validation_h_grid,
        half_width=settings["fft_half_width"], spatial_points=settings["fft_spatial_points"],
    )
    validation_absolute_error = float(np.max(np.abs(validation_fft - validation_exact)))
    validation_relative_error = validation_absolute_error / max(float(np.max(validation_exact)), np.finfo(float).tiny)
    print(f"FFT validation: maximum absolute D2 error={validation_absolute_error:.3e}; "
          f"error relative to max(D2)={validation_relative_error:.3e}")
    if validation_relative_error > 2e-5:
        raise ArithmeticError("FFT validation failed; increase the FFT resolution or interval before running.")


## Repeated spectral selections

The next cell computes the ten sample-size distributions for every configured lambda, sharing samples across both sample sizes and lambda values. A checkpoint is saved after every completed repetition. Rerunning with unchanged settings resumes that run.


In [ ]:
results_by_n = {}
for current_n in SAMPLE_SIZES:
    results_by_n[current_n] = run_l2_point(
        current_n, full_samples, h_grid, settings, LAMBDA_VALUES, run_directory,
        reuse=REUSE_CHECKPOINTS,
    )
print(f"Completed {len(SAMPLE_SIZES)} distributions per lambda for {LAMBDA_VALUES}.")


## Saved spectral selections and distributions

Export the chosen bandwidth for each repetition, the grid frequencies, and one summary row for each sample size and lambda. These files contain spectral selections only.


In [ ]:
selection_rows = []
frequency_rows = []
summary_rows = []
for current_n in SAMPLE_SIZES:
    result = results_by_n[current_n]
    for column, lambda_value in enumerate(LAMBDA_VALUES):
        chosen_indices = result["selected_indices"][:, column]
        chosen_h = h_grid[chosen_indices]
        selection_rows.extend(
            {"n": current_n, "lambda": lambda_value, "repetition": repetition,
             "grid_index": int(index), "h": float(h_grid[index])}
            for repetition, index in enumerate(chosen_indices)
        )
        frequency_rows.extend(
            {"n": current_n, "lambda": lambda_value, "grid_index": index,
             "h": float(h), "count": int(result["counts"][column, index]),
             "relative_frequency": float(result["frequencies"][column, index])}
            for index, h in enumerate(h_grid)
        )
        modal_index = int(np.argmax(result["counts"][column]))
        summary_rows.append({
            "n": current_n, "lambda": lambda_value, "repetitions": settings["repetitions"],
            "modal_h": float(h_grid[modal_index]), "mean_h": float(np.mean(chosen_h)),
            "median_h": float(np.median(chosen_h)),
            "modal_frequency": float(result["frequencies"][column, modal_index]),
        })
for filename, rows in (
    ("selections.csv", selection_rows), ("frequencies.csv", frequency_rows),
    ("summary.csv", summary_rows),
):
    with (run_directory / filename).open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
print(f"CSV data: {run_directory}")
print(f"{'lambda':>9s} {'n':>5s} {'modal h':>10s} {'mean h':>10s} {'median h':>10s}")
for row in sorted(summary_rows, key=lambda row: (row["lambda"], row["n"])):
    print(f"{row['lambda']:9g} {row['n']:5d} {row['modal_h']:10.5f} "
          f"{row['mean_h']:10.5f} {row['median_h']:10.5f}")


## First figure: ten spectral L² distributions at lambda = 1

The first displayed figure always contains the ten distributions at $\lambda=1$, arranged in a 5 × 2 grid. Each panel contains one spectral histogram. Subsequent lambda values, when configured, receive the same layout in increasing order. Horizontal and vertical limits are shared across all displayed figures.

Every displayed figure is automatically saved as a PDF under `outputs/pdfs/fast_l2_bandwidth_distributions_vs_n_lambda/`; a PNG copy is also saved. No ISE distribution, Q-hat distribution, or reference line is added.

Source for the experiment and geometric-grid plotting convention: [original Figure 3.6 workflow](spectral_scale_selection_kde.ipynb).


In [ ]:
if set(results_by_n) != set(SAMPLE_SIZES):
    raise RuntimeError("Finish all ten sample sizes before plotting.")
grid_positions = np.arange(len(h_grid))
ticks = np.unique(np.linspace(0, len(h_grid) - 1, 7, dtype=int))
y_max = 1.08 * max(np.max(result["frequencies"]) for result in results_by_n.values())

for column, lambda_value in enumerate(LAMBDA_VALUES):
    fig, axes = plt.subplots(5, 2, figsize=(14, 18), sharex=True, sharey=True)
    for ax, current_n in zip(axes.flat, SAMPLE_SIZES):
        ax.bar(
            grid_positions, results_by_n[current_n]["frequencies"][column],
            width=0.86, color="#D6843B", alpha=0.85, edgecolor="white", linewidth=0.25,
        )
        ax.set_title(rf"$n={current_n}$", fontsize=13, pad=8)
        ax.set_xticks(ticks, [f"{h_grid[index]:.3g}" for index in ticks])
        ax.set_xlim(-0.7, len(h_grid) - 0.3)
        ax.set_ylim(0, y_max)
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
        ax.set_xlabel("spectrally selected bandwidth h", fontsize=10)
        ax.set_ylabel("relative frequency", fontsize=10)
        ax.tick_params(labelsize=9, colors="#304967", labelbottom=True, labelleft=True)
        ax.grid(axis="y", color="#DDE5EC", linewidth=0.7)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color("#B8C5D1")
    fig.suptitle(
        rf"Fast $L^2$ spectral bandwidth distributions: $\lambda={lambda_value:g}$" + "\n"
        + rf"Gaussian data; $M={settings['repetitions']}$ per $n$, $N={len(h_grid)}$, "
        + r"$S=\exp(-\lambda D_2)$" + f" ({RUN_PROFILE})",
        fontsize=16, y=0.995,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.965), h_pad=2.0, w_pad=2.0)
    lambda_tag = f"{lambda_value:g}".replace(".", "p").replace("+", "")
    png_path = image_directory / (
        f"spectral_l2_lambda{lambda_tag}_M{settings['repetitions']}_N{len(h_grid)}.png"
    )
    fig.savefig(png_path, dpi=160, bbox_inches="tight")
    plt.show()  # The shared automatic exporter saves each ten-panel figure as a PDF.
    plt.close(fig)
    print(f"PNG (lambda={lambda_value:g}): {png_path}")
print(f"PDF directory: {pdf_output_directory}")
